# Tahap 4 — Advanced Normalization (UTS)
**Minggu 7–8 | Muhammad Iqbal Fadel (202310370311268)**

Tahap paling krusial: normalisasi kata gaul ke bahasa formal menggunakan Slang Dictionary domain parfum.

**Proses:**
1. Load kamus slang dari `dictionary/slang_dictionary_parfum.json`
2. Normalisasi huruf berulang (`wangiiii` → `wangi`)
3. Lookup token ke kamus → ganti dengan padanan formal

**Input:** `data/interim/ulasan_parfum_interim.csv`  
**Output:** `data/processed/ulasan_parfum_processed.csv`

In [ ]:
import sys, json
from pathlib import Path

ROOT_DIR = Path().resolve().parent.parent
sys.path.insert(0, str(ROOT_DIR))
print(f'Root: {ROOT_DIR}')

In [ ]:
# Lihat struktur kamus slang
dict_path = ROOT_DIR / 'dictionary' / 'slang_dictionary_parfum.json'
with open(dict_path, 'r', encoding='utf-8') as f:
    kamus_raw = json.load(f)

print('Kategori dalam kamus slang:')
total = 0
for kategori, isi in kamus_raw.items():
    if kategori.startswith('_'): continue
    print(f'  - {kategori}: {len(isi)} entri')
    total += len(isi)
print(f'\nTotal entri: {total}')

In [ ]:
# Demonstrasi fungsi normalisasi
import re

# Flatten kamus nested -> flat dict
kamus_flat = {}
for key, value in kamus_raw.items():
    if key.startswith('_'): continue
    if isinstance(value, dict):
        for slang, formal in value.items():
            kamus_flat[slang.lower()] = formal

def normalisasi_huruf_berulang(teks):
    return re.sub(r'(.)\1{2,}', r'\1', teks)

def slang_to_formal(teks, kamus):
    tokens = teks.split()
    hasil = [kamus.get(t, t) for t in tokens]
    hasil = [h for h in hasil if h]  # hapus string kosong (noise filler)
    return ' '.join(hasil)

def normalisasi_penuh(teks, kamus):
    teks = normalisasi_huruf_berulang(teks)
    teks = slang_to_formal(teks, kamus)
    return re.sub(r'\s+', ' ', teks).strip()

# Contoh transformasi
contoh = [
    ('ok bgt wanginya', 'ok bgt wanginya'),
    ('ga worth sih harganya', 'ga worth sih harganya'),
    ('rekomen bgt buat yg suka wangi soft', 'rekomen bgt buat yg suka wangi soft'),
]
print('Contoh Slang-to-Formal Normalization:')
print('-' * 60)
for raw, clean in contoh:
    normalized = normalisasi_penuh(clean, kamus_flat)
    print(f'  Raw       : {raw}')
    print(f'  Clean     : {clean}')
    print(f'  Normalized: {normalized}')
    print()

In [ ]:
# Jalankan pipeline normalisasi penuh
import importlib.util

script = ROOT_DIR / 'scripts' / '04_advanced_normalization' / 'normalization.py'
spec = importlib.util.spec_from_file_location('normalization', script)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

mod.main()

In [ ]:
# Lihat hasil normalisasi
import pandas as pd
df = pd.read_csv(ROOT_DIR / 'data' / 'processed' / 'ulasan_parfum_processed.csv')
print(f'Total baris: {len(df):,}')
print(f'Kolom: {list(df.columns)}')
display(df[['REVIEW', 'text_clean', 'text_normalized']].head(10))

## Hasil

- Dataset ternormalisasi tersimpan di: `data/processed/ulasan_parfum_processed.csv`
- Kamus: 120 entri, 10 kategori domain parfum
- Coverage kamus: ~0,56% token (dataset dominan Bahasa Inggris)